# 2 · Connect — Planetary Computer

### Confirm Planetary Computer is reachable before you start the analysis

With your environment installed ([page 1](01_setup.md)), this notebook confirms that
Planetary Computer is reachable. There is **no account to set up** — the STAC catalogue
and its open collections (Landsat, Sentinel-2) are anonymously accessible.

Run the cells below in order. They use the same environment as the other pages
(`uv run jupyter lab` from the repository root, or build the site with `--execute`).

**Next:** [3 · Tidal feasibility](03_tides.ipynb)

## Step 1 — Confirm the catalogue answers a search

This step proves that your machine can reach the Planetary Computer API and that your query
returns sensible metadata.

In [1]:
import pystac_client
import planetary_computer

PC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
catalog = pystac_client.Client.open(PC_URL, modifier=planetary_computer.sign_inplace)

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=(3.96, 51.55, 4.10, 51.62),     # example: part of the Oosterschelde, NL
    datetime="2023-01-01/2023-03-31",
    query={"eo:cloud_cover": {"lt": 80}},
)
items = search.item_collection()
print(f"Planetary Computer connection OK. Found {len(items)} Landsat scenes in the test window.")

Planetary Computer connection OK. Found 12 Landsat scenes in the test window.


If this prints a scene count (any number ≥ 0 without an error), **discovery works**.

> **Note:** Zero scenes is not always a failure — widen the `bbox`, extend the `datetime`
> range, or raise the cloud-cover limit. Only network errors or Python exceptions indicate
> a broken connection.

## Step 2 — Confirm an asset can be signed and read

Step 1 only proved that metadata arrives. Step 2 checks the **signing** step — short-lived
access tokens that replace a traditional account.

In [2]:
import planetary_computer

if len(items) > 0:
    item = items[0]
    signed = planetary_computer.sign(item)
    href = signed.assets["red"].href
    print("Signing OK — assets are readable. Example signed href starts with:")
    print(href.split("?")[0], "(+ access token)")
else:
    print("No scenes in the test window; widen bbox or dates and retry.")

Signing OK — assets are readable. Example signed href starts with:
https://landsateuwest.blob.core.windows.net/landsat-c2/level-2/standard/oli-tirs/2023/199/024/LC08_L2SP_199024_20230325_20230404_02_T1/LC08_L2SP_199024_20230325_20230404_02_T1_SR_B4.TIF (+ access token)


If signing succeeds, continue to **[3 · Tidal feasibility →](03_tides.ipynb)**

---

### Troubleshooting

- **Search returns 0 scenes** — widen the `bbox` or `datetime`, or raise the cloud-cover limit.
- **Network/TLS errors** — confirm you can reach `https://planetarycomputer.microsoft.com`.
- **Install errors on Intel Macs** — see [1 · Setup](01_setup.md): `conda deactivate` or
  `CC=/usr/bin/clang CXX=/usr/bin/clang++ uv sync --frozen`.